# CDS Hydrology Forecast Dataset: Download and Plot

Dataset: `sis-hydrology-variables-derived-seasonal-forecast`

This notebook:
- downloads one subset from CDS via `cdsapi`
- opens the NetCDF with `xarray`
- plots a map and a point time series

If the request fails due to parameter mismatch, copy the exact API request from the CDS **Download** tab and paste it in the request cell below.

In [9]:
from __future__ import annotations

from pathlib import Path
import os
import zipfile

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

%matplotlib widget

import cdsapi

matplotlib.rcParams["text.usetex"] = False
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["font.sans-serif"] = ["DejaVu Sans", "Liberation Sans", "Arial"]
matplotlib.rcParams["mathtext.fontset"] = "dejavusans"

def find_project_root(start: Path) -> Path:
    p = start.resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "data").exists() and (candidate / "src").exists():
            return candidate
    return p

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "data" / "cds"
FIG_DIR = ROOT / "figures"
DATA_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT: {ROOT}")


def resolve_cdsapirc(root: Path) -> Path | None:
    candidates = [
        root / '.cdsapirc',
        Path.cwd() / '.cdsapirc',
        Path.home() / '.cdsapirc',
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

CDSAPI_RC_PATH = resolve_cdsapirc(ROOT)
if CDSAPI_RC_PATH is not None:
    os.environ['CDSAPI_RC'] = str(CDSAPI_RC_PATH)


print(f"CDSAPI_RC selected: {CDSAPI_RC_PATH}")


ROOT: /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis
CDSAPI_RC selected: /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/.cdsapirc


## Request parameters

The default request below is a **starter template**.
If CDS rejects it, replace `REQUEST` with the exact script from the CDS Download tab.

In [10]:
DATASET_ID = "sis-hydrology-variables-derived-seasonal-forecast"
FORCE_DOWNLOAD = False  # set True once to refresh for a new year/month selection

# Valid minimal request (adjust after first successful run)
REQUEST = {
    "variable": ["river_discharge"],
    "hydrological_model": ["lisflood_efas"],
    "year": ["2021", "2022", "2023", "2024"],
    "month": ["06"],
    "version": ["1"],
}

years_tag = f"{REQUEST['year'][0]}-{REQUEST['year'][-1]}" if len(REQUEST['year']) > 1 else REQUEST['year'][0]
months_tag = '-'.join(REQUEST['month'])
TARGET_FILE = DATA_DIR / f"cds_hydrology_forecast_{years_tag}_m{months_tag}.zip"
NETCDF_FILE = DATA_DIR / f"cds_hydrology_forecast_{years_tag}_m{months_tag}.nc"

print(f"Dataset: {DATASET_ID}")
print(f"Target archive: {TARGET_FILE}")
print(f"Preferred NetCDF path: {NETCDF_FILE}")
print("Request keys:", list(REQUEST.keys()))


Dataset: sis-hydrology-variables-derived-seasonal-forecast
Target archive: /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/cds_hydrology_forecast_2021-2024_m06.zip
Preferred NetCDF path: /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/cds_hydrology_forecast_2021-2024_m06.nc
Request keys: ['variable', 'hydrological_model', 'year', 'month', 'version']


In [11]:
if (TARGET_FILE.exists() or NETCDF_FILE.exists()) and not FORCE_DOWNLOAD:
    print(f"Using existing data in {DATA_DIR}")
else:
    if CDSAPI_RC_PATH is None:
        raise RuntimeError("Missing .cdsapirc. Put it in project root or home directory.")

    print("Submitting CDS request...")
    client = cdsapi.Client()
    try:
        client.retrieve(DATASET_ID, REQUEST, str(TARGET_FILE))
        print(f"Download complete: {TARGET_FILE}")
    except Exception as exc:
        msg = str(exc)
        if "required licences not accepted" in msg.lower():
            raise RuntimeError(
                "CDS account is authenticated but dataset licence is not accepted yet.\n"
                "Open the dataset page and accept licences:\n"
                "https://cds.climate.copernicus.eu/datasets/sis-hydrology-variables-derived-seasonal-forecast?tab=download#manage-licences"
            )
        raise RuntimeError(
            "CDS request failed. If parameters changed, copy the exact API request from CDS Download tab into REQUEST.\n"
            f"Original error: {exc}"
        )

DS_FILES = []
if zipfile.is_zipfile(TARGET_FILE):
    with zipfile.ZipFile(TARGET_FILE) as zf:
        members = sorted(name for name in zf.namelist() if name.endswith(".nc"))
        if not members:
            raise RuntimeError(f"Downloaded ZIP contains no NetCDF files: {TARGET_FILE}")
        for member in members:
            zf.extract(member, path=DATA_DIR)
            extracted = DATA_DIR / member
            final_path = DATA_DIR / Path(member).name
            if extracted != final_path:
                extracted.replace(final_path)
            DS_FILES.append(final_path)
else:
    DS_FILES = [NETCDF_FILE if NETCDF_FILE.exists() else TARGET_FILE]

print(f"Dataset files for xarray ({len(DS_FILES)}):")
for p in DS_FILES:
    print(' -', p)


Submitting CDS request...


2026-04-25 16:29:45,243 INFO Request ID is ddb04ae9-6f2a-4213-b7ab-25abbee0ef1a
2026-04-25 16:29:45,304 INFO status has been updated to accepted
2026-04-25 16:30:06,416 INFO status has been updated to running
2026-04-25 16:32:37,838 INFO status has been updated to successful


e051136233d56ae862fd21938382e5ff.zip:   0%|          | 0.00/3.83G [00:00<?, ?B/s]

Download complete: /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/cds_hydrology_forecast_2021-2024_m06.zip
Dataset files for xarray (3):
 - /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/rdis_seas5_EFAS_20210601_v1.nc
 - /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/rdis_seas5_EFAS_20220601_v1.nc
 - /home/vsilv/.nextcloud/src/cassini/11th_cassini_hackathon/python_analysis/data/cds/rdis_seas5_EFAS_20230601_v1.nc


In [12]:
if len(DS_FILES) == 1:
    ds = xr.open_dataset(DS_FILES[0], engine="netcdf4")
else:
    ds = xr.open_mfdataset([str(p) for p in DS_FILES], engine="netcdf4", combine="by_coords")
print(ds)
print("\nVariables:", list(ds.data_vars))

ImportError: chunk manager 'dask' is not available. Please make sure 'dask' is installed and importable.

In [ ]:
# Alpine-focused future maps
PLOT_VARIABLE = "rdis"
POINT_LAT = 46.8
POINT_LON = 11.3
ALPS_BBOX = {"lat_min": 43.0, "lat_max": 48.8, "lon_min": 5.0, "lon_max": 16.5}
MAX_LEADS_TO_PLOT = 7

y_name = next((c for c in ["y", "lat", "latitude"] if c in ds.dims), None)
x_name = next((c for c in ["x", "lon", "longitude"] if c in ds.dims), None)
if y_name is None or x_name is None:
    raise RuntimeError(f"Could not detect spatial dimensions. Found dims={list(ds.dims)}")

lat_coord = "latitude" if "latitude" in ds.coords else ("lat" if "lat" in ds.coords else y_name)
lon_coord = "longitude" if "longitude" in ds.coords else ("lon" if "lon" in ds.coords else x_name)

if PLOT_VARIABLE not in ds.data_vars:
    PLOT_VARIABLE = None
if PLOT_VARIABLE is None:
    for name, da_try in ds.data_vars.items():
        if y_name in da_try.dims and x_name in da_try.dims:
            PLOT_VARIABLE = name
            break
if PLOT_VARIABLE is None:
    raise RuntimeError("No plottable geospatial variable found. Set PLOT_VARIABLE manually.")

da = ds[PLOT_VARIABLE]
member_dim = next((d for d in ["member", "number", "ensemble_member"] if d in da.dims), None)
time_candidates = ["valid_time", "time", "step", "forecast_reference_time", "t"]
time_name = next((t for t in time_candidates if t in da.dims or t in da.coords or t in ds.coords), None)
if time_name is None:
    raise RuntimeError(f"Could not detect time axis in dims={da.dims} and coords={list(ds.coords)}")
if time_name in da.dims:
    time_dim = time_name
elif time_name in da.coords and da[time_name].ndim > 0:
    time_dim = da[time_name].dims[0]
elif time_name in ds.coords and ds[time_name].ndim > 0:
    time_dim = ds[time_name].dims[0]
else:
    time_dim = da.dims[0]

da_map = da.mean(dim=member_dim, skipna=True) if member_dim else da

if lat_coord in ds.coords and lon_coord in ds.coords and ds[lat_coord].ndim == 2 and ds[lon_coord].ndim == 2:
    lat2d = ds[lat_coord]
    lon2d = ds[lon_coord]
else:
    lon2d, lat2d = np.meshgrid(ds[x_name].values, ds[y_name].values)

alps_mask = (lat2d >= ALPS_BBOX["lat_min"]) & (lat2d <= ALPS_BBOX["lat_max"]) & (lon2d >= ALPS_BBOX["lon_min"]) & (lon2d <= ALPS_BBOX["lon_max"])
if bool(np.all(~alps_mask)):
    raise RuntimeError("Alpine bbox does not overlap dataset grid.")

n_leads = min(da_map.sizes[time_dim], MAX_LEADS_TO_PLOT)
ncols = 4
nrows = int(np.ceil(n_leads / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.5 * ncols, 3.8 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
im = None

for i in range(n_leads):
    ax = axes[i]
    da_i = da_map.isel({time_dim: i}).where(alps_mask)
    im = ax.pcolormesh(lon2d, lat2d, da_i, shading="auto", cmap="viridis")

    if "valid_time" in ds.coords and ds["valid_time"].ndim > 0:
        vt = ds["valid_time"].isel({ds["valid_time"].dims[0]: i}).values
        label = np.datetime_as_string(np.datetime64(vt), unit="D")
    else:
        label = f"{time_dim}={da_map[time_dim].isel({time_dim: i}).values}"

    ax.set_title(label)
    ax.set_xlim(ALPS_BBOX["lon_min"], ALPS_BBOX["lon_max"])
    ax.set_ylim(ALPS_BBOX["lat_min"], ALPS_BBOX["lat_max"])
    ax.grid(alpha=0.2)

for j in range(n_leads, len(axes)):
    axes[j].axis("off")

fig.suptitle(f"Alpine future simulations: {PLOT_VARIABLE} (ensemble mean)", y=1.02)
fig.supxlabel("longitude")
fig.supylabel("latitude")
if im is not None:
    fig.colorbar(im, ax=axes[:n_leads], shrink=0.9, label=PLOT_VARIABLE)
plt.tight_layout()
plt.show()

In [ ]:
# Alpine point future ensemble time series
if lat_coord in ds.coords and lon_coord in ds.coords and ds[lat_coord].ndim == 2 and ds[lon_coord].ndim == 2:
    dist2 = (ds[lat_coord] - POINT_LAT) ** 2 + (ds[lon_coord] - POINT_LON) ** 2
    iy, ix = np.unravel_index(np.nanargmin(dist2.values), dist2.shape)
    da_point = ds[PLOT_VARIABLE].isel({y_name: int(iy), x_name: int(ix)})
else:
    da_point = ds[PLOT_VARIABLE].sel({y_name: POINT_LAT, x_name: POINT_LON}, method="nearest")

if time_name in da_point.dims:
    ts_time_dim = time_name
elif time_name in da_point.coords and da_point[time_name].ndim > 0:
    ts_time_dim = da_point[time_name].dims[0]
else:
    ts_time_dim = da_point.dims[0]

member_dim_local = next((d for d in ["member", "number", "ensemble_member"] if d in da_point.dims), None)
keep_dims = [ts_time_dim] + ([member_dim_local] if member_dim_local else [])
reduce_dims = [d for d in da_point.dims if d not in keep_dims]
da_point = da_point.mean(dim=reduce_dims, skipna=True) if reduce_dims else da_point

if "valid_time" in ds.coords and ds["valid_time"].ndim > 0:
    xvals = ds["valid_time"].values
    x_label = "valid_time"
elif time_name in da_point.coords:
    xvals = da_point[time_name].values
    x_label = time_name
else:
    xvals = da_point[ts_time_dim].values
    x_label = ts_time_dim

fig, ax = plt.subplots(figsize=(10, 4.5))
if member_dim_local:
    for i in range(da_point.sizes[member_dim_local]):
        ax.plot(xvals, da_point.isel({member_dim_local: i}).values, color="tab:blue", alpha=0.12, linewidth=0.8)
    mean_ts = da_point.mean(dim=member_dim_local, skipna=True)
    p10_ts = da_point.quantile(0.1, dim=member_dim_local, skipna=True)
    p90_ts = da_point.quantile(0.9, dim=member_dim_local, skipna=True)
    ax.fill_between(xvals, p10_ts.values, p90_ts.values, color="tab:blue", alpha=0.25, label="P10-P90")
    ax.plot(xvals, mean_ts.values, color="black", linewidth=2.0, label="Ensemble mean")
else:
    ax.plot(xvals, da_point.values, marker="o", linewidth=1.8, color="black")

ax.set_title(f"Future simulations at Alpine point ({POINT_LAT:.2f}, {POINT_LON:.2f})")
ax.set_xlabel(x_label)
ax.set_ylabel(PLOT_VARIABLE)
ax.grid(alpha=0.3)
if member_dim_local:
    ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Same calendar month by year (Alpine maps)
TARGET_MONTH = 6
TARGET_YEARS = [2021, 2022, 2023, 2024]

da_dec = ds[PLOT_VARIABLE]
member_dim = next((d for d in ["member", "number", "ensemble_member"] if d in da_dec.dims), None)
if member_dim:
    da_dec = da_dec.mean(dim=member_dim, skipna=True)

time_candidates = ["valid_time", "time", "step", "forecast_reference_time", "t"]
time_name = next((t for t in time_candidates if t in da_dec.coords or t in ds.coords or t in da_dec.dims), None)
if time_name is None:
    raise RuntimeError("Could not detect a time coordinate for year-by-year plotting.")

if time_name in da_dec.coords:
    tcoord = da_dec[time_name]
elif time_name in ds.coords:
    tcoord = ds[time_name]
else:
    tcoord = da_dec[da_dec.dims[0]]

if not np.issubdtype(tcoord.dtype, np.datetime64):
    raise RuntimeError(
        f"Time coordinate '{time_name}' is not datetime-like (dtype={tcoord.dtype}). "
        "Need datetime values to build yearly month panels."
    )

if time_name in da_dec.dims:
    time_dim = time_name
elif tcoord.ndim > 0:
    time_dim = tcoord.dims[0]
else:
    raise RuntimeError("Time coordinate has no dimension; cannot build yearly means.")

months = tcoord.dt.month
years = tcoord.dt.year
same_month = da_dec.where(months == TARGET_MONTH, drop=True)
if same_month.sizes.get(time_dim, 0) == 0:
    raise RuntimeError(f"No samples found for month={TARGET_MONTH}.")

available_years = np.unique(years.where(months == TARGET_MONTH, drop=True).values.astype(int))
missing_years = [y for y in TARGET_YEARS if y not in available_years]
if missing_years:
    raise RuntimeError(
        f"Missing requested years for month={TARGET_MONTH}: {missing_years}. "
        "Set FORCE_DOWNLOAD=True and rerun the download cell with REQUEST['year'] covering these years."
    )

if lat_coord in ds.coords and lon_coord in ds.coords and ds[lat_coord].ndim == 2 and ds[lon_coord].ndim == 2:
    lat2d = ds[lat_coord]
    lon2d = ds[lon_coord]
else:
    lon2d, lat2d = np.meshgrid(ds[x_name].values, ds[y_name].values)

alps_mask = (lat2d >= ALPS_BBOX["lat_min"]) & (lat2d <= ALPS_BBOX["lat_max"]) & (lon2d >= ALPS_BBOX["lon_min"]) & (lon2d <= ALPS_BBOX["lon_max"])

n_panels = len(TARGET_YEARS)
ncols = 4
nrows = int(np.ceil(n_panels / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4.6 * ncols, 3.8 * nrows), sharex=True, sharey=True)
axes = np.atleast_1d(axes).ravel()
im = None

for i, year in enumerate(TARGET_YEARS):
    ax = axes[i]
    year_mask = (years == year) & (months == TARGET_MONTH)
    da_plot = da_dec.where(year_mask, drop=True).mean(dim=time_dim, skipna=True).where(alps_mask)

    im = ax.pcolormesh(lon2d, lat2d, da_plot, shading="auto", cmap="viridis")
    ax.set_title(str(year))
    ax.set_xlim(ALPS_BBOX["lon_min"], ALPS_BBOX["lon_max"])
    ax.set_ylim(ALPS_BBOX["lat_min"], ALPS_BBOX["lat_max"])
    ax.grid(alpha=0.2)

for j in range(n_panels, len(axes)):
    axes[j].axis("off")

fig.suptitle(f"Alpine {PLOT_VARIABLE}: June mean by year (ensemble mean)", y=1.02)
fig.supxlabel("longitude")
fig.supylabel("latitude")
if im is not None:
    fig.colorbar(im, ax=axes[:n_panels], shrink=0.9, label=PLOT_VARIABLE)
plt.tight_layout()
plt.show()